# GRPO + vLLM Validation and Performance Baseline

This notebook is the final confirmation flow for rollout-side validation.

You will validate:

1. Baseline BEFORE weight sync (latency/TTFT/throughput).
2. Deterministic weight-sync correctness (unsynced -> synced).
3. Baseline AFTER weight sync.
4. Rollout control-plane health (`/health`, `/get_world_size`, `/pause`, `/resume`).


## Runtime auto-detection (usually no edits)

This notebook is designed to run directly in an OpenShift AI workbench with minimal setup.

Auto-detected by default:

- namespace (`/var/run/secrets/kubernetes.io/serviceaccount/namespace`, then `NOTEBOOK_NAMESPACE`, then `oc project -q`)
- workbench pod (`HOSTNAME` when in-pod, otherwise the newest running workbench-labeled pod in the namespace)
- notebook container (first container in the selected workbench pod)

Optional overrides are available in the next cell for advanced or out-of-cluster runs, but most runs should be plug-and-play.


In [ ]:
from __future__ import annotations

import datetime as dt
import json
import os
import re
import shlex
import subprocess
import time
from pathlib import Path
from typing import Any


def find_grpo_root() -> Path:
    env_root = os.getenv('GRPO_TRAINER_ROOT')
    search_roots: list[Path] = []
    if env_root:
        search_roots.append(Path(env_root).expanduser().resolve())

    cwd = Path.cwd().resolve()
    search_roots.extend([cwd, *cwd.parents])

    for base in search_roots:
        direct_validate = base / 'scripts' / 'validate_cold_start_weight_sync.sh'
        direct_baseline = base / 'scripts' / 'capture_vllm_performance_baseline.sh'
        if direct_validate.exists() and direct_baseline.exists():
            return base

        nested = base / 'research' / 'grpo-trainer'
        nested_validate = nested / 'scripts' / 'validate_cold_start_weight_sync.sh'
        nested_baseline = nested / 'scripts' / 'capture_vllm_performance_baseline.sh'
        if nested_validate.exists() and nested_baseline.exists():
            return nested

    raise FileNotFoundError(
        'Could not locate grpo-trainer root. Set GRPO_TRAINER_ROOT or run this notebook from inside the repo.'
    )


ROOT = find_grpo_root()
SCRIPT_VALIDATE = ROOT / 'scripts' / 'validate_cold_start_weight_sync.sh'
SCRIPT_BASELINE = ROOT / 'scripts' / 'capture_vllm_performance_baseline.sh'

RUN_TS = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
OUTPUT_ROOT = ROOT / 'runs_log' / f'notebook-final-validation-{RUN_TS}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Research root: {ROOT}')
print(f'Run output dir: {OUTPUT_ROOT}')


def run(cmd: str, *, check: bool = True, env: dict | None = None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update({k: str(v) for k, v in env.items()})
    print(f"\n$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        cwd=str(ROOT),
        text=True,
        capture_output=True,
        env=merged_env,
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    print(f'[exit_code={proc.returncode}]')
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {cmd}')
    return proc


def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matching {pattern} in {directory}')
    return matches[-1]


def extract_metrics(payload: dict[str, Any]) -> dict[str, Any]:
    s = payload.get('signals', {})
    return {
        'latency_ms_p50': s.get('latency_ms_p50'),
        'latency_ms_p95': s.get('latency_ms_p95'),
        'ttft_ms_p50': s.get('ttft_ms_p50'),
        'ttft_ms_p95': s.get('ttft_ms_p95'),
        'throughput_rps': s.get('throughput_rps'),
        'token_throughput_tokens_per_s': s.get('token_throughput_tokens_per_s'),
        'error_rate': s.get('error_rate'),
    }


def pct_delta(before: float | None, after: float | None):
    if before in (None, 0) or after is None:
        return None
    return ((after - before) / before) * 100.0


def detect_namespace() -> str:
    # Preferred order: service account namespace file -> NOTEBOOK_NAMESPACE -> current oc project
    sa_ns_path = Path('/var/run/secrets/kubernetes.io/serviceaccount/namespace')
    if sa_ns_path.exists():
        ns = sa_ns_path.read_text().strip()
        if ns:
            return ns

    env_ns = os.getenv('NOTEBOOK_NAMESPACE', '').strip()
    if env_ns:
        return env_ns

    proc = run('oc project -q', check=False)
    ns = (proc.stdout or '').strip()
    if proc.returncode == 0 and ns:
        return ns

    raise RuntimeError(
        'Could not auto-detect namespace. Ensure oc login/project is configured or set NOTEBOOK_NAMESPACE.'
    )


def _list_pods(namespace: str) -> list[dict[str, Any]]:
    proc = run(f'oc get pods -n {shlex.quote(namespace)} -o json')
    payload = json.loads(proc.stdout or '{}')
    return payload.get('items', [])


def detect_workbench_pod(namespace: str, override: str | None = None) -> tuple[str, str]:
    if override:
        run(f'oc get pod {shlex.quote(override)} -n {shlex.quote(namespace)}')
        return override, 'override'

    hostname = os.getenv('HOSTNAME', '').strip()
    if hostname:
        probe = run(
            f'oc get pod {shlex.quote(hostname)} -n {shlex.quote(namespace)}',
            check=False,
        )
        if probe.returncode == 0:
            return hostname, 'HOSTNAME'

    pods = _list_pods(namespace)
    running = [p for p in pods if p.get('status', {}).get('phase') == 'Running']

    def is_workbench(pod: dict[str, Any]) -> bool:
        md = pod.get('metadata', {})
        labels = md.get('labels', {}) or {}
        name = md.get('name', '').lower()
        if 'notebook-name' in labels:
            return True
        component = (labels.get('app.kubernetes.io/component') or '').lower()
        app_name = (labels.get('app.kubernetes.io/name') or '').lower()
        dashboard = labels.get('opendatahub.io/dashboard')
        if component in {'workbench', 'notebook'}:
            return True
        if app_name in {'workbench', 'notebook'}:
            return True
        if dashboard is not None:
            return True
        return 'workbench' in name or 'notebook' in name

    workbench_candidates = [p for p in running if is_workbench(p)]

    def sort_key(pod: dict[str, Any]) -> str:
        return pod.get('metadata', {}).get('creationTimestamp', '')

    if workbench_candidates:
        selected = sorted(workbench_candidates, key=sort_key, reverse=True)[0]
        return selected['metadata']['name'], 'workbench-label-discovery'

    if running:
        selected = sorted(running, key=sort_key, reverse=True)[0]
        return selected['metadata']['name'], 'running-pod-fallback'

    raise RuntimeError(f'No running pods found in namespace {namespace}.')


def detect_container(namespace: str, pod_name: str, override: str | None = None) -> str:
    if override:
        return override

    proc = run(
        f"oc get pod {shlex.quote(pod_name)} -n {shlex.quote(namespace)} -o jsonpath='{{.spec.containers[0].name}}'"
    )
    container = (proc.stdout or '').strip()
    if not container:
        raise RuntimeError(f'Could not detect container for pod {pod_name}.')
    return container


def derive_workbench_name(namespace: str, pod_name: str) -> str:
    pod = run(
        f'oc get pod {shlex.quote(pod_name)} -n {shlex.quote(namespace)} -o json',
        check=False,
    )
    if pod.returncode == 0:
        payload = json.loads(pod.stdout or '{}')
        labels = payload.get('metadata', {}).get('labels', {}) or {}
        if labels.get('notebook-name'):
            return labels['notebook-name']

    if '-' in pod_name and pod_name.rsplit('-', 1)[-1].isdigit():
        return pod_name.rsplit('-', 1)[0]
    return pod_name


In [ ]:
# Optional overrides (leave unset for auto-detect)
OVERRIDE_NAMESPACE = os.getenv('VALIDATION_NAMESPACE') or None
OVERRIDE_NOTEBOOK_POD = (
    os.getenv('VALIDATION_NOTEBOOK_POD')
    or os.getenv('WORKBENCH_POD_NAME')
    or None
)
OVERRIDE_NOTEBOOK_CONTAINER = os.getenv('VALIDATION_NOTEBOOK_CONTAINER') or None

# Rollout defaults
VLLM_SERVICE = os.getenv('VLLM_SERVICE', 'grpo-vllm-rollout')
VLLM_DEPLOYMENT = os.getenv('VLLM_DEPLOYMENT', VLLM_SERVICE)
VLLM_SELECTOR = os.getenv('VLLM_SELECTOR', f'app={VLLM_SERVICE}')
MODEL_NAME = os.getenv('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')

# Baseline and validation controls
BASELINE_REQUESTS = int(os.getenv('BASELINE_REQUESTS', '30'))
BASELINE_MAX_TOKENS = int(os.getenv('BASELINE_MAX_TOKENS', '64'))
VALIDATION_ATTEMPTS = int(os.getenv('VALIDATION_ATTEMPTS', '3'))
VALIDATION_SKIP_RESTART = os.getenv('VALIDATION_SKIP_RESTART', 'false').lower() == 'true'

# Training signal validation controls

# Auto-resolve runtime context
NAMESPACE = OVERRIDE_NAMESPACE or detect_namespace()
NOTEBOOK_POD, NOTEBOOK_POD_SOURCE = detect_workbench_pod(NAMESPACE, OVERRIDE_NOTEBOOK_POD)
NOTEBOOK_CONTAINER = detect_container(NAMESPACE, NOTEBOOK_POD, OVERRIDE_NOTEBOOK_CONTAINER)
NOTEBOOK_NAME = derive_workbench_name(NAMESPACE, NOTEBOOK_POD)

print('Resolved namespace:', NAMESPACE)
print('Resolved notebook pod:', NOTEBOOK_POD, f'(source={NOTEBOOK_POD_SOURCE})')
print('Resolved notebook container:', NOTEBOOK_CONTAINER)
print('Resolved notebook name:', NOTEBOOK_NAME)
print('Model:', MODEL_NAME)

In [ ]:
# Preflight checks + rollout readiness gate
run(f'oc project {shlex.quote(NAMESPACE)}')
run(f'oc get pod {shlex.quote(NOTEBOOK_POD)} -n {shlex.quote(NAMESPACE)}')
run(f'oc get svc {shlex.quote(VLLM_SERVICE)} -n {shlex.quote(NAMESPACE)}')
run(f'oc get deployment {shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)}')
selector_probe = run(
    f'oc get pods -n {shlex.quote(NAMESPACE)} -l {shlex.quote(VLLM_SELECTOR)} -o name',
    check=False,
)
if selector_probe.returncode != 0:
    print('Warning: vLLM selector probe failed. Continue if selector is intentionally custom.')

service_host = f"{VLLM_SERVICE}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"


def _probe_service(path: str) -> bool:
    url = f"{service_root}{path}"
    cmd = (
        f"oc exec -n {shlex.quote(NAMESPACE)} {shlex.quote(NOTEBOOK_POD)} "
        f"-c {shlex.quote(NOTEBOOK_CONTAINER)} -- "
        f"curl -sS -m 10 -o /dev/null -w '%{{http_code}}' {shlex.quote(url)}"
    )
    probe = run(cmd, check=False)
    code = ''.join(ch for ch in (probe.stdout or '') if ch.isdigit())
    return probe.returncode == 0 and code.endswith('200')


def wait_for_rollout_readiness(timeout_s: int = 180, interval_s: int = 10) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        health_ok = _probe_service('/health')
        world_ok = _probe_service('/get_world_size')
        if health_ok and world_ok:
            return True
        print('Rollout service not ready yet; retrying...')
        time.sleep(interval_s)
    return False


ready = wait_for_rollout_readiness()
if not ready:
    print('Rollout still not ready; restarting rollout deployment once...')
    run(f'oc rollout restart deployment/{shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)}')
    run(f'oc rollout status deployment/{shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)} --timeout=600s')
    ready = wait_for_rollout_readiness(timeout_s=240, interval_s=10)

if not ready:
    raise RuntimeError(
        'Rollout service is not reachable from the workbench pod after restart. '
        'Check network policy, DNS, and rollout pod logs.'
    )

print('Notebook context is ready for baseline + validation.')
print('Service root:', service_root)
print('Scripts present:', SCRIPT_VALIDATE.exists(), SCRIPT_BASELINE.exists())


## 1) Baseline BEFORE weight sync

This captures rollout performance before validation/sync.

In [ ]:
def run_baseline(tag: str):
    out_dir = OUTPUT_ROOT / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = (
        f"{shlex.quote(str(SCRIPT_BASELINE))} "
        f"--namespace {shlex.quote(NAMESPACE)} "
        f"--notebook-pod {shlex.quote(NOTEBOOK_POD)} "
        f"--notebook-container {shlex.quote(NOTEBOOK_CONTAINER)} "
        f"--vllm-service {shlex.quote(VLLM_SERVICE)} "
        f"--vllm-selector {shlex.quote(VLLM_SELECTOR)} "
        f"--model-name {shlex.quote(MODEL_NAME)} "
        f"--requests {int(BASELINE_REQUESTS)} "
        f"--max-tokens {int(BASELINE_MAX_TOKENS)} "
        f"--output-dir {shlex.quote(str(out_dir))}"
    )

    last_proc = None
    for attempt in range(1, 3):
        last_proc = run(cmd, check=False)
        if last_proc.returncode == 0:
            break

        combined = f"{last_proc.stdout or ''}\n{last_proc.stderr or ''}"
        transient_not_found = 'Error from server (NotFound): pods "' in combined

        if attempt < 2 and transient_not_found:
            print('Baseline probe hit transient pod-not-found; retrying after rollout pod refresh...')
            run(
                f'oc get pods -n {shlex.quote(NAMESPACE)} -l {shlex.quote(VLLM_SELECTOR)} -o wide',
                check=False,
            )
            time.sleep(5)
            continue

        raise RuntimeError(f'Baseline capture failed (attempt={attempt}): {cmd}')

    if last_proc is None or last_proc.returncode != 0:
        raise RuntimeError(f'Baseline capture did not succeed: {cmd}')

    raw_path = latest_file(out_dir, 'vllm-baseline-raw-*.json')
    summary_path = latest_file(out_dir, 'vllm-baseline-summary-*.md')
    data = json.loads(raw_path.read_text())
    metrics = extract_metrics(data)

    print('\n=== BASELINE', tag.upper(), '===')
    print('raw:', raw_path)
    print('summary:', summary_path)
    print(json.dumps(metrics, indent=2))

    return {
        'tag': tag,
        'out_dir': out_dir,
        'raw': raw_path,
        'summary': summary_path,
        'payload': data,
        'metrics': metrics,
    }

BEFORE = run_baseline('before')

## 2) Deterministic weight-sync validation

This proves unsynced -> synced transition and prints sync evidence directly in output.

In [ ]:
skip_flag = '--skip-restart' if VALIDATION_SKIP_RESTART else ''
validate_cmd = (
    f"{shlex.quote(str(SCRIPT_VALIDATE))} "
    f"--namespace {shlex.quote(NAMESPACE)} "
    f"--notebook-pod {shlex.quote(NOTEBOOK_POD)} "
    f"--notebook-container {shlex.quote(NOTEBOOK_CONTAINER)} "
    f"--vllm-deployment {shlex.quote(VLLM_DEPLOYMENT)} "
    f"--vllm-service {shlex.quote(VLLM_SERVICE)} "
    f"--vllm-selector {shlex.quote(VLLM_SELECTOR)} "
    f"--model-name {shlex.quote(MODEL_NAME)} "
    f"--attempts {int(VALIDATION_ATTEMPTS)} "
    f"{skip_flag}"
).strip()

run(validate_cmd)
print('\nWeight sync validation: PASS')


## 3) Baseline AFTER weight sync

This captures post-sync performance using the same parameters.

In [ ]:
AFTER = run_baseline('after')


In [ ]:
print('=== PERFORMANCE COMPARISON (after vs before) ===')
rows = [
    'latency_ms_p50',
    'latency_ms_p95',
    'ttft_ms_p50',
    'ttft_ms_p95',
    'throughput_rps',
    'token_throughput_tokens_per_s',
    'error_rate',
]

for k in rows:
    b = BEFORE['metrics'].get(k)
    a = AFTER['metrics'].get(k)
    d = pct_delta(b, a)
    delta = 'n/a' if d is None else f'{d:+.2f}%'
    print(f'- {k}: before={b} | after={a} | delta={delta}')


## 4) Rollout service health + control-plane checks

This cell validates health and control-plane endpoints on the rollout service.

In [ ]:
import urllib.error
import urllib.request

service_host = f"{VLLM_SERVICE}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
print('Rollout service root:', service_root)


def _curl_fallback(path: str, method: str):
    url = f"{service_root}{path}"
    method_arg = '' if method == 'GET' else f'-X {method} '
    cmd = (
        f"oc exec -n {shlex.quote(NAMESPACE)} {shlex.quote(NOTEBOOK_POD)} "
        f"-c {shlex.quote(NOTEBOOK_CONTAINER)} -- "
        f"curl -sS -w '\n%{{http_code}}' {method_arg}{shlex.quote(url)}"
    )
    proc = run(cmd, check=False)
    out = (proc.stdout or '').strip()
    lines = out.splitlines() if out else []
    status = None
    body = ''
    if lines:
        maybe_code = lines[-1].strip()
        if maybe_code.isdigit():
            status = int(maybe_code)
            body = '\n'.join(lines[:-1])
        else:
            body = out
    return status, body, proc.returncode


def check_endpoint(path: str, method: str = 'GET'):
    url = f"{service_root}{path}"
    try:
        req = urllib.request.Request(url=url, method=method)
        with urllib.request.urlopen(req, timeout=30) as resp:
            body = resp.read().decode('utf-8', errors='replace')
            print(f"{method} {path}: PASS (direct) status={resp.status}")
            print(body[:300])
            return True
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError) as exc:
        print(f"{method} {path}: direct request failed ({exc}); trying oc exec fallback")

    status, body, rc = _curl_fallback(path, method)
    if rc == 0 and status is not None and 200 <= status < 300:
        print(f"{method} {path}: PASS (oc exec fallback) status={status}")
        print(body[:300])
        return True

    print(f"{method} {path}: FAIL (rc={rc}, status={status})")
    if body:
        print(body[:500])
    return False


checks = [
    ('GET', '/health'),
    ('GET', '/get_world_size'),
    ('POST', '/pause'),
    ('POST', '/resume'),
]

failures = []
for method, path in checks:
    ok = check_endpoint(path=path, method=method)
    if not ok:
        failures.append(f"{method} {path}")

if failures:
    raise RuntimeError('Rollout service checks failed: ' + ', '.join(failures))

print('Rollout service health/control-plane checks: PASS')


## Final interpretation

Use this notebook as a focused rollout validation package:

- **Step 1 (Baseline before):** Rollout latency/throughput captured before weight sync.
- **Step 2 (Weight sync):** If `validate_cold_start_weight_sync.sh` exits 0 and the sync probe converges, the unsynced -> synced transition is confirmed.
- **Step 3 (Baseline after):** Rollout performance captured after sync. Compare against Step 1.
- **Step 4 (Control-plane):** `/health`, `/get_world_size`, `/pause`, `/resume` all return 2xx.

Run artifacts are written under:

`research/grpo-trainer/runs_log/notebook-final-validation-<timestamp>/`
